Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\24pasos_gru_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 7)
Dimensiones de Y: (43765, 1)


In [9]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 7)
Las dimensiones de testX son:  (8797, 12, 7)
Las dimensiones de valX son:  (4333, 12, 7)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

49/49 - 13s - 265ms/step - ia: 0.3520 - loss: 1.6214 - mae: 0.8818 - rmse: 1.2628 - smape: 1.2645 - val_ia: 0.2965 - val_loss: 0.5803 - val_mae: 0.5190 - val_rmse: 0.6672 - val_smape: 1.0246

Epoch 2/128                                           

49/49 - 1s - 22ms/step - ia: 0.3224 - loss: 1.4818 - mae: 0.8552 - rmse: 1.2026 - smape: 1.3343 - val_ia: 0.2832 - val_loss: 0.5388 - val_mae: 0.5233 - val_rmse: 0.6585 - val_smape: 1.1548

Epoch 3/128                                           

49/49 - 1s - 24ms/step - ia: 0.2909 - loss: 1.3817 - mae: 0.8452 - rmse: 1.1630 - smape: 1.3970 - val_ia: 0.2712 - val_loss: 0.5309 - val_mae: 0.5397 - val_rmse: 0.6678 - val_smape: 1.3266

Epoch 4/128                                           

49/49 - 1s - 19ms/step - ia: 0.2693 - loss: 1.3650 - mae: 0.8519 - rmse: 1.1574 - smape: 1.4370 - val_ia: 0.2639 - val_loss: 0.5349 - val_mae: 0.5531 - val_rmse: 0.6779 - val_smape: 1.4643

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



385/385 - 27s - 71ms/step - ia: 0.3961 - loss: 0.9109 - mae: 0.7048 - rmse: 0.9254 - smape: 1.2388 - val_ia: 0.2447 - val_loss: 0.4506 - val_mae: 0.4973 - val_rmse: 0.5546 - val_smape: 1.2350

Epoch 2/128                                                                     

385/385 - 8s - 21ms/step - ia: 0.4341 - loss: 0.8567 - mae: 0.6753 - rmse: 0.8931 - smape: 1.1843 - val_ia: 0.2579 - val_loss: 0.4607 - val_mae: 0.4935 - val_rmse: 0.5518 - val_smape: 1.1792

Epoch 3/128                                                                     

385/385 - 9s - 23ms/step - ia: 0.4433 - loss: 0.8322 - mae: 0.6644 - rmse: 0.8826 - smape: 1.1664 - val_ia: 0.2451 - val_loss: 0.4922 - val_mae: 0.5170 - val_rmse: 0.5783 - val_smape: 1.2782

Epoch 4/128                                                                     

385/385 - 8s - 22ms/step - ia: 0.4477 - loss: 0.8088 - mae: 0.6556 - rmse: 0.8702 - smape: 1.1665 - val_ia: 0.2642 - val_loss: 0.4750 - val_mae: 0.4928 - val_rmse: 0.5567 - val_

...

Epoch 1/128                                                                      

385/385 - 17s - 44ms/step - ia: 0.2772 - loss: 1.5242 - mae: 0.9441 - rmse: 1.1984 - smape: 1.4667 - val_ia: 0.2391 - val_loss: 0.5276 - val_mae: 0.5636 - val_rmse: 0.6184 - val_smape: 1.6861

Epoch 2/128                                                                      

385/385 - 6s - 15ms/step - ia: 0.2889 - loss: 1.2862 - mae: 0.8559 - rmse: 1.1008 - smape: 1.4403 - val_ia: 0.2526 - val_loss: 0.4930 - val_mae: 0.5240 - val_rmse: 0.5794 - val_smape: 1.3362

Epoch 3/128                                                                      

385/385 - 9s - 23ms/step - ia: 0.2958 - loss: 1.1820 - mae: 0.8145 - rmse: 1.0512 - smape: 1.4304 - val_ia: 0.2546 - val_loss: 0.4725 - val_mae: 0.5103 - val_rmse: 0.5654 - val_smape: 1.2872

Epoch 4/128                                                                      

385/385 - 6s - 16ms/step - ia: 0.3167 - loss: 1.1009 - mae: 0.7924 - rmse: 1.0171 - smape: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



770/770 - 17s - 23ms/step - ia: 0.3150 - loss: 1.6075 - mae: 0.8910 - rmse: 1.1743 - smape: 1.3693 - val_ia: 0.2322 - val_loss: 0.6344 - val_mae: 0.5467 - val_rmse: 0.5852 - val_smape: 1.0713

Epoch 2/128                                                                       

770/770 - 6s - 8ms/step - ia: 0.3138 - loss: 1.5743 - mae: 0.8800 - rmse: 1.1599 - smape: 1.3668 - val_ia: 0.2323 - val_loss: 0.6286 - val_mae: 0.5441 - val_rmse: 0.5826 - val_smape: 1.0739

Epoch 3/128                                                                       

770/770 - 6s - 8ms/step - ia: 0.3121 - loss: 1.5849 - mae: 0.8842 - rmse: 1.1649 - smape: 1.3835 - val_ia: 0.2319 - val_loss: 0.6228 - val_mae: 0.5416 - val_rmse: 0.5800 - val_smape: 1.0769

Epoch 4/128                                                                       

770/770 - 6s - 8ms/step - ia: 0.3128 - loss: 1.5607 - mae: 0.8731 - rmse: 1.1559 - smape: 1.3841 - val_ia: 0.2312 - val_loss: 0.6176 - val_mae: 0.5395 - val_rmse: 0.5778 - v

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

49/49 - 9s - 187ms/step - ia: 0.2507 - loss: 1.2081 - mae: 0.8288 - rmse: 1.0955 - smape: 1.4839 - val_ia: 0.2744 - val_loss: 0.4982 - val_mae: 0.5457 - val_rmse: 0.6655 - val_smape: 1.5540

Epoch 2/128                                                                       

49/49 - 2s - 36ms/step - ia: 0.3105 - loss: 1.0923 - mae: 0.7843 - rmse: 1.0343 - smape: 1.4011 - val_ia: 0.2853 - val_loss: 0.4549 - val_mae: 0.4971 - val_rmse: 0.6241 - val_smape: 1.2395

Epoch 3/128                                                                       

49/49 - 1s - 30ms/step - ia: 0.3606 - loss: 1.0204 - mae: 0.7579 - rmse: 1.0029 - smape: 1.3315 - val_ia: 0.2961 - val_loss: 0.4423 - val_mae: 0.4963 - val_rmse: 0.6221 - val_smape: 1.2542

Epoch 4/128                                                                       

49/49 - 2s - 48ms/step - ia: 0.3913 - loss: 0.9884 - mae: 0.7446 - rmse: 1.0099 - smape: 1.28

In [16]:
print(best)

{'activation': 1, 'batch': 4, 'dropout': 0.2, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}


In [17]:
#{'activation': 3, 'batch': 4, 'dropout': 0.30000000000000004, 'layers': 1.0, 'learning_rate': 0.001217652235385441, 'units': 0}